In [10]:
import requests
import json
import time
from pathlib import Path

BASE_URL = "https://raw.githubusercontent.com/statsbomb/open-data/master/data"

COMPETITION_ID = 11   # La Liga
SEASON_ID = 90 # 2020/2021

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "corners_dataset.jsonl"

SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "corner-extraction-module1/1.0"})

In [ ]:
def fetch_json(url: str, retries: int = 3, backoff: float = 1.5):
    """GET a JSON file with basic retry. Returns None on 404 (file legitimately absent,
    e.g. a match with no 360 data) rather than raising."""
    for attempt in range(retries):
        resp = SESSION.get(url, timeout=30)
        if resp.status_code == 404:
            return None
        if resp.status_code == 200:
            return resp.json()
        time.sleep(backoff * (attempt + 1))
    resp.raise_for_status()

def get_matches(competition_id: int, season_id: int):
    url = f"{BASE_URL}/matches/{competition_id}/{season_id}.json"
    matches = fetch_json(url)
    if matches is None:
        raise ValueError(f"No matches file found for competition={competition_id}, season={season_id}")
    return matches

def get_events(match_id: int):
    url = f"{BASE_URL}/events/{match_id}.json"
    return fetch_json(url)

def get_360(match_id: int):
    url = f"{BASE_URL}/three-sixty/{match_id}.json"
    return fetch_json(url) 

In [16]:
matches = get_matches(COMPETITION_ID, SEASON_ID)
print(f"Found {len(matches)} matches for competition={COMPETITION_ID}, season={SEASON_ID}")

match_ids = [m["match_id"] for m in matches]
print("Example match record keys:", list(matches[0].keys())[:10])
print("First few match IDs:", match_ids[:5])

Found 35 matches for competition=11, season=90
Example match record keys: ['match_id', 'match_date', 'kick_off', 'competition', 'season', 'home_team', 'away_team', 'home_score', 'away_score', 'match_status']
First few match IDs: [3773386, 3773565, 3773457, 3773631, 3773665]


In [19]:
def is_corner(event: dict) -> bool:
    return (
        event.get("type", {}).get("name") == "Pass"
        and event.get("pass", {}).get("type", {}).get("name") == "Corner"
    )


def classify_corner_sequence(events_by_index: dict, corner: dict):
    """Returns (label, chain) where chain is the ordered list of events in the
    corner's possession that occur after the corner itself."""
    possession_id = corner["possession"]
    attacking_team_id = corner["team"]["id"]
    corner_idx = corner["index"]

    chain = sorted(
        (e for e in events_by_index.values()
         if e["possession"] == possession_id and e["index"] > corner_idx),
        key=lambda e: e["index"],
    )

    pass_count = 0
    for event in chain:
        type_name = event["type"]["name"]
        if type_name == "Shot" and event["team"]["id"] == attacking_team_id:
            if pass_count == 0:
                return "direct", chain
            elif pass_count <= 3:
                return f"{pass_count}-pass", chain
            else:
                return "4-plus-pass", chain
        if type_name == "Pass" and event["team"]["id"] == attacking_team_id:
            pass_count += 1

    return "no-shot", chain

In [20]:
from collections import Counter

test_match_id = match_ids[0]
test_events = get_events(test_match_id)
test_events_by_index = {e["index"]: e for e in test_events}
test_corners = [e for e in test_events if is_corner(e)]

print(f"Match {test_match_id}: {len(test_events)} events, {len(test_corners)} corners")

labels = []
for c in test_corners:
    label, chain = classify_corner_sequence(test_events_by_index, c)
    labels.append(label)
    print(f"  corner idx={c['index']:>5}  minute={c['minute']:>3}  team={c['team']['name']:<20}  ->  {label}")

print("\nLabel distribution:", Counter(labels))

Match 3773386: 3891 events, 16 corners
  corner idx=  394  minute=  8  team=Deportivo Alavés      ->  no-shot
  corner idx= 1002  minute= 25  team=Deportivo Alavés      ->  no-shot
  corner idx= 1699  minute= 42  team=Barcelona             ->  no-shot
  corner idx= 2005  minute= 48  team=Barcelona             ->  no-shot
  corner idx= 2455  minute= 57  team=Barcelona             ->  no-shot
  corner idx= 2687  minute= 64  team=Barcelona             ->  no-shot
  corner idx= 3046  minute= 74  team=Barcelona             ->  no-shot
  corner idx= 3415  minute= 83  team=Barcelona             ->  1-pass
  corner idx= 3618  minute= 87  team=Barcelona             ->  no-shot
  corner idx= 3633  minute= 88  team=Barcelona             ->  3-pass
  corner idx= 3721  minute= 90  team=Barcelona             ->  1-pass
  corner idx= 3730  minute= 90  team=Barcelona             ->  no-shot
  corner idx= 3740  minute= 91  team=Barcelona             ->  no-shot
  corner idx= 3749  minute= 91  team=Barc

In [21]:
def get_freeze_frame(frames_by_uuid: dict, corner_event_id: str):
    frame = frames_by_uuid.get(corner_event_id)
    return frame["freeze_frame"] if frame else None

In [22]:
test_frames = get_360(test_match_id)
test_frames_by_uuid = {f["event_uuid"]: f for f in test_frames} if test_frames else {}

print(f"Match {test_match_id}: {len(test_frames)} total 360 frames")

n_with_frame = 0
for c in test_corners:
    ff = get_freeze_frame(test_frames_by_uuid, c["id"])
    has_it = ff is not None
    n_with_frame += has_it
    print(f"  corner idx={c['index']:>5}  has_freeze_frame={has_it}")

print(f"\n{n_with_frame}/{len(test_corners)} corners have a matching freeze frame")

Match 3773386: 3670 total 360 frames
  corner idx=  394  has_freeze_frame=True
  corner idx= 1002  has_freeze_frame=True
  corner idx= 1699  has_freeze_frame=False
  corner idx= 2005  has_freeze_frame=False
  corner idx= 2455  has_freeze_frame=True
  corner idx= 2687  has_freeze_frame=False
  corner idx= 3046  has_freeze_frame=False
  corner idx= 3415  has_freeze_frame=True
  corner idx= 3618  has_freeze_frame=False
  corner idx= 3633  has_freeze_frame=False
  corner idx= 3721  has_freeze_frame=True
  corner idx= 3730  has_freeze_frame=True
  corner idx= 3740  has_freeze_frame=True
  corner idx= 3749  has_freeze_frame=False
  corner idx= 3840  has_freeze_frame=True
  corner idx= 3880  has_freeze_frame=True

9/16 corners have a matching freeze frame


In [23]:
def build_corner_record(match, corner, chain, label, freeze_frame):
    return {
        "match_id": match["match_id"],
        "competition_id": COMPETITION_ID,
        "season_id": SEASON_ID,
        "match_date": match.get("match_date"),
        "home_team": match["home_team"]["home_team_name"],
        "away_team": match["away_team"]["away_team_name"],
        "corner_event_id": corner["id"],
        "corner_index": corner["index"],
        "period": corner["period"],
        "minute": corner["minute"],
        "second": corner["second"],
        "attacking_team_id": corner["team"]["id"],
        "attacking_team_name": corner["team"]["name"],
        "taker_player_id": corner.get("player", {}).get("id"),
        "taker_player_name": corner.get("player", {}).get("name"),
        "corner_location": corner.get("location"),
        "pass_end_location": corner.get("pass", {}).get("end_location"),
        "pass_technique": corner.get("pass", {}).get("technique", {}).get("name"),
        "pass_outcome": corner.get("pass", {}).get("outcome", {}).get("name"),
        "recipient_player_id": corner.get("pass", {}).get("recipient", {}).get("id"),
        "recipient_player_name": corner.get("pass", {}).get("recipient", {}).get("name"),
        "label": label,
        "chain_event_types": [e["type"]["name"] for e in chain],
        "has_freeze_frame": freeze_frame is not None,
        "freeze_frame": freeze_frame,
    }

In [25]:
test_records = []
for corner in test_corners:
    label, chain = classify_corner_sequence(test_events_by_index, corner)
    freeze_frame = get_freeze_frame(test_frames_by_uuid, corner["id"])
    record = build_corner_record(matches[0], corner, chain, label, freeze_frame)
    test_records.append(record)

print(f"Built {len(test_records)} records")
print(json.dumps({k: v for k, v in test_records[7].items() if k != "freeze_frame"}, indent=2))
print(f"freeze_frame present: {test_records[7]['has_freeze_frame']}")

Built 16 records
{
  "match_id": 3773386,
  "competition_id": 11,
  "season_id": 90,
  "match_date": "2020-10-31",
  "home_team": "Deportivo Alav\u00e9s",
  "away_team": "Barcelona",
  "corner_event_id": "24c6c5fd-4557-4b15-b8a8-6acc7d30d51c",
  "corner_index": 3415,
  "period": 2,
  "minute": 83,
  "second": 0,
  "attacking_team_id": 217,
  "attacking_team_name": "Barcelona",
  "taker_player_id": 30486,
  "taker_player_name": "Pedro Gonz\u00e1lez L\u00f3pez",
  "corner_location": [
    120.0,
    0.1
  ],
  "pass_end_location": [
    110.1,
    4.1
  ],
  "pass_technique": null,
  "pass_outcome": null,
  "recipient_player_id": 6947,
  "recipient_player_name": "Miralem Pjani\u0107",
  "label": "1-pass",
  "chain_event_types": [
    "Ball Receipt*",
    "Carry",
    "Pass",
    "Ball Receipt*",
    "Carry",
    "Shot",
    "Block",
    "Goal Keeper",
    "Pass",
    "Ball Receipt*",
    "Carry",
    "Pass",
    "Ball Receipt*",
    "Pass",
    "Ball Receipt*",
    "Carry",
    "Pressure

In [26]:
FINAL_THIRD_X = 80  # StatsBomb pitch is 120 long; x=120 is the opponent's goal line

def is_freekick_pass(event: dict) -> bool:
    if event.get("type", {}).get("name") != "Pass":
        return False
    if event.get("pass", {}).get("type", {}).get("name") != "Free Kick":
        return False
    x, _ = event.get("location", [0, 0])
    return x >= FINAL_THIRD_X  # exclude deep restarts, keep box-delivery free kicks


def is_freekick_direct_shot(event: dict) -> bool:
    return (
        event.get("type", {}).get("name") == "Shot"
        and event.get("shot", {}).get("type", {}).get("name") == "Free Kick"
    )

In [27]:
test_fk_passes_all = [e for e in test_events
                       if e.get("type", {}).get("name") == "Pass"
                       and e.get("pass", {}).get("type", {}).get("name") == "Free Kick"]
test_fk_passes_dangerous = [e for e in test_events if is_freekick_pass(e)]
test_fk_shots = [e for e in test_events if is_freekick_direct_shot(e)]

print(f"Total free-kick passes in match: {len(test_fk_passes_all)}")
print(f"  -> after final-third filter (x>=80): {len(test_fk_passes_dangerous)}")
print(f"Direct free-kick shots: {len(test_fk_shots)}")

print("\n--- Dangerous free-kick passes, classified same way as corners ---")
fk_labels = []
for fk in test_fk_passes_dangerous:
    label, chain = classify_corner_sequence(test_events_by_index, fk)
    fk_labels.append(label)
    print(f"  idx={fk['index']:>5}  minute={fk['minute']:>3}  team={fk['team']['name']:<20}  ->  {label}")

print("\n--- Direct free-kick shots (always label='direct', no chain to walk) ---")
for s in test_fk_shots:
    print(f"  idx={s['index']:>5}  minute={s['minute']:>3}  team={s['team']['name']:<20}  ->  direct")

print("\nLabel distribution (dangerous FK passes only):", Counter(fk_labels))

Total free-kick passes in match: 20
  -> after final-third filter (x>=80): 3
Direct free-kick shots: 4

--- Dangerous free-kick passes, classified same way as corners ---
  idx= 2778  minute= 67  team=Barcelona             ->  direct
  idx= 3578  minute= 87  team=Barcelona             ->  direct
  idx= 3713  minute= 90  team=Barcelona             ->  no-shot

--- Direct free-kick shots (always label='direct', no chain to walk) ---
  idx=  929  minute= 22  team=Barcelona             ->  direct
  idx= 1696  minute= 42  team=Barcelona             ->  direct
  idx= 1726  minute= 44  team=Deportivo Alavés      ->  direct
  idx= 2501  minute= 59  team=Barcelona             ->  direct

Label distribution (dangerous FK passes only): Counter({'direct': 2, 'no-shot': 1})


Cell 10: run corner extraction over the whole season and save

In [28]:
all_records = []
match_stats = []

for i, match in enumerate(matches):
    mid = match["match_id"]
    events = get_events(mid)
    if not events:
        print(f"[{i+1}/{len(matches)}] match {mid}: no events file, skipping")
        continue

    events_by_index = {e["index"]: e for e in events}
    corners = [e for e in events if is_corner(e)]

    frames = get_360(mid)
    frames_by_uuid = {f["event_uuid"]: f for f in frames} if frames else {}

    n_with_frame = 0
    for corner in corners:
        label, chain = classify_corner_sequence(events_by_index, corner)
        freeze_frame = get_freeze_frame(frames_by_uuid, corner["id"])
        if freeze_frame is not None:
            n_with_frame += 1
        record = build_corner_record(match, corner, chain, label, freeze_frame)
        all_records.append(record)

    match_stats.append({"match_id": mid, "n_corners": len(corners), "n_with_freeze_frame": n_with_frame})
    print(f"[{i+1}/{len(matches)}] match {mid}: {len(corners)} corners, {n_with_frame} with freeze frame")

print(f"\nTotal corners collected: {len(all_records)}")

[1/35] match 3773386: 16 corners, 9 with freeze frame
[2/35] match 3773565: 5 corners, 4 with freeze frame
[3/35] match 3773457: 8 corners, 5 with freeze frame
[4/35] match 3773631: 8 corners, 7 with freeze frame
[5/35] match 3773665: 7 corners, 7 with freeze frame
[6/35] match 3773497: 11 corners, 9 with freeze frame
[7/35] match 3773660: 16 corners, 14 with freeze frame
[8/35] match 3773593: 12 corners, 8 with freeze frame
[9/35] match 3773466: 10 corners, 8 with freeze frame
[10/35] match 3773585: 14 corners, 10 with freeze frame
[11/35] match 3773552: 8 corners, 6 with freeze frame
[12/35] match 3773672: 11 corners, 8 with freeze frame
[13/35] match 3773587: 9 corners, 7 with freeze frame
[14/35] match 3773656: 16 corners, 12 with freeze frame
[15/35] match 3773377: 11 corners, 9 with freeze frame
[16/35] match 3773586: 7 corners, 5 with freeze frame
[17/35] match 3773372: 12 corners, 10 with freeze frame
[18/35] match 3773387: 14 corners, 12 with freeze frame
[19/35] match 3773695

Cell: save the corner dataset to disk

In [29]:
with open(OUTPUT_FILE, "w") as f:
    for record in all_records:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(all_records)} records to {OUTPUT_FILE.resolve()}")

Saved 329 records to D:\Report\Uni\L4S1\Code\output\corners_dataset.jsonl


In [30]:
label_counts = Counter(r["label"] for r in all_records)
freeze_frame_coverage = sum(r["has_freeze_frame"] for r in all_records) / len(all_records)

print("Label distribution across the season:")
for label, count in label_counts.most_common():
    print(f"  {label:<15} {count:>4}  ({count/len(all_records):.1%})")

print(f"\nFreeze-frame coverage: {freeze_frame_coverage:.1%} of corners")
print(f"Usable-for-graph corners (has freeze frame): {sum(r['has_freeze_frame'] for r in all_records)}")

Label distribution across the season:
  no-shot          218  (66.3%)
  direct            45  (13.7%)
  4-plus-pass       21  (6.4%)
  1-pass            17  (5.2%)
  3-pass            14  (4.3%)
  2-pass            14  (4.3%)

Freeze-frame coverage: 76.0% of corners
Usable-for-graph corners (has freeze frame): 250


Cell: build the free-kick record function

In [31]:
def build_freekick_record(match, event, chain, label, freeze_frame, set_piece_type):
    is_shot_type = event["type"]["name"] == "Shot"

    record = {
        "match_id": match["match_id"],
        "competition_id": COMPETITION_ID,
        "season_id": SEASON_ID,
        "match_date": match.get("match_date"),
        "home_team": match["home_team"]["home_team_name"],
        "away_team": match["away_team"]["away_team_name"],
        "set_piece_type": set_piece_type,  # "freekick_delivery" or "freekick_direct_shot"
        "event_id": event["id"],
        "event_index": event["index"],
        "period": event["period"],
        "minute": event["minute"],
        "second": event["second"],
        "attacking_team_id": event["team"]["id"],
        "attacking_team_name": event["team"]["name"],
        "taker_player_id": event.get("player", {}).get("id"),
        "taker_player_name": event.get("player", {}).get("name"),
        "location": event.get("location"),
        "label": label,
        "chain_event_types": [e["type"]["name"] for e in chain],
        "has_freeze_frame": freeze_frame is not None,
        "freeze_frame": freeze_frame,
    }

    if is_shot_type:
        record["shot_outcome"] = event.get("shot", {}).get("outcome", {}).get("name")
        record["shot_xg"] = event.get("shot", {}).get("statsbomb_xg")
        record["pass_end_location"] = None
        record["recipient_player_id"] = None
        record["recipient_player_name"] = None
    else:
        record["shot_outcome"] = None
        record["shot_xg"] = None
        record["pass_end_location"] = event.get("pass", {}).get("end_location")
        record["recipient_player_id"] = event.get("pass", {}).get("recipient", {}).get("id")
        record["recipient_player_name"] = event.get("pass", {}).get("recipient", {}).get("name")

    return record

In [32]:
all_freekick_records = []
fk_match_stats = []

for i, match in enumerate(matches):
    mid = match["match_id"]
    events = get_events(mid)
    if not events:
        print(f"[{i+1}/{len(matches)}] match {mid}: no events file, skipping")
        continue

    events_by_index = {e["index"]: e for e in events}
    fk_deliveries = [e for e in events if is_freekick_pass(e)]
    fk_direct_shots = [e for e in events if is_freekick_direct_shot(e)]

    frames = get_360(mid)
    frames_by_uuid = {f["event_uuid"]: f for f in frames} if frames else {}

    n_with_frame = 0

    for fk in fk_deliveries:
        label, chain = classify_corner_sequence(events_by_index, fk)  # same chain logic as corners
        freeze_frame = get_freeze_frame(frames_by_uuid, fk["id"])
        if freeze_frame is not None:
            n_with_frame += 1
        record = build_freekick_record(match, fk, chain, label, freeze_frame, "freekick_delivery")
        all_freekick_records.append(record)

    for shot in fk_direct_shots:
        freeze_frame = get_freeze_frame(frames_by_uuid, shot["id"])
        if freeze_frame is not None:
            n_with_frame += 1
        record = build_freekick_record(match, shot, [], "direct", freeze_frame, "freekick_direct_shot")
        all_freekick_records.append(record)

    n_total = len(fk_deliveries) + len(fk_direct_shots)
    fk_match_stats.append({"match_id": mid, "n_freekicks": n_total, "n_with_freeze_frame": n_with_frame})
    print(f"[{i+1}/{len(matches)}] match {mid}: {len(fk_deliveries)} deliveries, {len(fk_direct_shots)} direct shots, {n_with_frame} with freeze frame")

print(f"\nTotal free kicks collected: {len(all_freekick_records)}")

[1/35] match 3773386: 3 deliveries, 4 direct shots, 6 with freeze frame
[2/35] match 3773565: 2 deliveries, 2 direct shots, 4 with freeze frame
[3/35] match 3773457: 5 deliveries, 0 direct shots, 4 with freeze frame
[4/35] match 3773631: 3 deliveries, 1 direct shots, 3 with freeze frame
[5/35] match 3773665: 3 deliveries, 0 direct shots, 3 with freeze frame
[6/35] match 3773497: 4 deliveries, 5 direct shots, 9 with freeze frame
[7/35] match 3773660: 5 deliveries, 3 direct shots, 7 with freeze frame
[8/35] match 3773593: 1 deliveries, 1 direct shots, 2 with freeze frame
[9/35] match 3773466: 4 deliveries, 3 direct shots, 6 with freeze frame
[10/35] match 3773585: 3 deliveries, 1 direct shots, 4 with freeze frame
[11/35] match 3773552: 2 deliveries, 2 direct shots, 4 with freeze frame
[12/35] match 3773672: 3 deliveries, 1 direct shots, 3 with freeze frame
[13/35] match 3773587: 6 deliveries, 2 direct shots, 7 with freeze frame
[14/35] match 3773656: 3 deliveries, 1 direct shots, 4 with 

In [33]:
FREEKICK_OUTPUT_FILE = OUTPUT_DIR / "freekicks_dataset.jsonl"

with open(FREEKICK_OUTPUT_FILE, "w") as f:
    for record in all_freekick_records:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(all_freekick_records)} records to {FREEKICK_OUTPUT_FILE.resolve()}")

Saved 164 records to D:\Report\Uni\L4S1\Code\output\freekicks_dataset.jsonl


In [34]:
fk_label_counts = Counter(r["label"] for r in all_freekick_records)
fk_type_counts = Counter(r["set_piece_type"] for r in all_freekick_records)
fk_freeze_coverage = sum(r["has_freeze_frame"] for r in all_freekick_records) / len(all_freekick_records)

print("Set-piece type breakdown:")
for t, count in fk_type_counts.most_common():
    print(f"  {t:<22} {count:>4}  ({count/len(all_freekick_records):.1%})")

print("\nLabel distribution:")
for label, count in fk_label_counts.most_common():
    print(f"  {label:<15} {count:>4}  ({count/len(all_freekick_records):.1%})")

print(f"\nFreeze-frame coverage: {fk_freeze_coverage:.1%}")
print(f"Usable-for-graph free kicks: {sum(r['has_freeze_frame'] for r in all_freekick_records)}")

Set-piece type breakdown:
  freekick_delivery       110  (67.1%)
  freekick_direct_shot     54  (32.9%)

Label distribution:
  no-shot           88  (53.7%)
  direct            66  (40.2%)
  4-plus-pass        3  (1.8%)
  3-pass             3  (1.8%)
  2-pass             2  (1.2%)
  1-pass             2  (1.2%)

Freeze-frame coverage: 87.8%
Usable-for-graph free kicks: 144


In [35]:
COMPETITIONS = [
    (9, 281),     # Bundesliga 2023/24
    (1267, 107),  # African Cup of Nations 2023
    (43, 106),    # FIFA World Cup 2022
    (11, 90),     # La Liga 2020/21
    (7, 235),     # Ligue 1 2022/23
    (7, 108),     # Ligue 1 2021/22
    (44, 107),    # MLS 2023
    (55, 282),    # UEFA Euro 2024
    (55, 43),     # UEFA Euro 2020
    (53, 315),    # UEFA Women's Euro 2025
    (53, 106),    # UEFA Women's Euro 2022
    (72, 107),    # Women's World Cup 2023
]

In [36]:
def extract_match(match, competition_id, season_id):
    """Returns (corner_records, freekick_records, error) for one match.
    error is None on success, or a string describing what went wrong."""
    mid = match["match_id"]
    corner_records, freekick_records = [], []

    try:
        events = get_events(mid)
        if not events:
            return [], [], "no events file"

        events_by_index = {e["index"]: e for e in events}
        frames = get_360(mid)
        frames_by_uuid = {f["event_uuid"]: f for f in frames} if frames else {}

        corners = [e for e in events if is_corner(e)]
        for corner in corners:
            label, chain = classify_corner_sequence(events_by_index, corner)
            freeze_frame = get_freeze_frame(frames_by_uuid, corner["id"])
            record = build_corner_record(match, corner, chain, label, freeze_frame)
            record["competition_id"] = competition_id  # override prototype-scope constants
            record["season_id"] = season_id
            corner_records.append(record)

        fk_deliveries = [e for e in events if is_freekick_pass(e)]
        for fk in fk_deliveries:
            label, chain = classify_corner_sequence(events_by_index, fk)
            freeze_frame = get_freeze_frame(frames_by_uuid, fk["id"])
            record = build_freekick_record(match, fk, chain, label, freeze_frame, "freekick_delivery")
            record["competition_id"] = competition_id
            record["season_id"] = season_id
            freekick_records.append(record)

        fk_shots = [e for e in events if is_freekick_direct_shot(e)]
        for shot in fk_shots:
            freeze_frame = get_freeze_frame(frames_by_uuid, shot["id"])
            record = build_freekick_record(match, shot, [], "direct", freeze_frame, "freekick_direct_shot")
            record["competition_id"] = competition_id
            record["season_id"] = season_id
            freekick_records.append(record)

        return corner_records, freekick_records, None

    except Exception as exc:
        return [], [], str(exc)

In [37]:
FULL_CORNERS_FILE = OUTPUT_DIR / "corners_dataset_full.jsonl"
FULL_FREEKICKS_FILE = OUTPUT_DIR / "freekicks_dataset_full.jsonl"

failed_matches = []
total_corners = 0
total_freekicks = 0
match_counter = 0

with open(FULL_CORNERS_FILE, "w") as cf, open(FULL_FREEKICKS_FILE, "w") as ff:
    for competition_id, season_id in COMPETITIONS:
        comp_matches = get_matches(competition_id, season_id)
        print(f"\n=== competition={competition_id} season={season_id}: {len(comp_matches)} matches ===")

        for match in comp_matches:
            match_counter += 1
            corner_records, freekick_records, error = extract_match(match, competition_id, season_id)

            if error:
                failed_matches.append({"match_id": match["match_id"], "competition_id": competition_id,
                                        "season_id": season_id, "error": error})
                print(f"  [{match_counter}] match {match['match_id']}: FAILED -- {error}")
                continue

            for record in corner_records:
                cf.write(json.dumps(record) + "\n")
            for record in freekick_records:
                ff.write(json.dumps(record) + "\n")
            cf.flush()
            ff.flush()

            total_corners += len(corner_records)
            total_freekicks += len(freekick_records)
            print(f"  [{match_counter}] match {match['match_id']}: "
                  f"{len(corner_records)} corners, {len(freekick_records)} free kicks "
                  f"(running totals: {total_corners} corners, {total_freekicks} free kicks)")

print(f"\n\nDONE. Total corners: {total_corners}, total free kicks: {total_freekicks}")
print(f"Failed matches: {len(failed_matches)}")
if failed_matches:
    print(json.dumps(failed_matches, indent=2))


=== competition=9 season=281: 34 matches ===
  [1] match 3895292: 17 corners, 6 free kicks (running totals: 17 corners, 6 free kicks)
  [2] match 3895320: 7 corners, 5 free kicks (running totals: 24 corners, 11 free kicks)
  [3] match 3895158: 17 corners, 4 free kicks (running totals: 41 corners, 15 free kicks)
  [4] match 3895107: 15 corners, 3 free kicks (running totals: 56 corners, 18 free kicks)
  [5] match 3895340: 8 corners, 1 free kicks (running totals: 64 corners, 19 free kicks)
  [6] match 3895286: 11 corners, 3 free kicks (running totals: 75 corners, 22 free kicks)
  [7] match 3895302: 7 corners, 0 free kicks (running totals: 82 corners, 22 free kicks)
  [8] match 3895333: 15 corners, 1 free kicks (running totals: 97 corners, 23 free kicks)
  [9] match 3895348: 9 corners, 4 free kicks (running totals: 106 corners, 27 free kicks)
  [10] match 3895220: 7 corners, 2 free kicks (running totals: 113 corners, 29 free kicks)
  [11] match 3895250: 13 corners, 9 free kicks (running t

In [38]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

full_corners = load_jsonl(FULL_CORNERS_FILE)
full_freekicks = load_jsonl(FULL_FREEKICKS_FILE)

print(f"Corners: {len(full_corners)} total, {sum(r['has_freeze_frame'] for r in full_corners)} usable "
      f"({sum(r['has_freeze_frame'] for r in full_corners)/len(full_corners):.1%})")
print(f"Free kicks: {len(full_freekicks)} total, {sum(r['has_freeze_frame'] for r in full_freekicks)} usable "
      f"({sum(r['has_freeze_frame'] for r in full_freekicks)/len(full_freekicks):.1%})")

print("\nCorner label distribution:")
for label, count in Counter(r["label"] for r in full_corners).most_common():
    print(f"  {label:<15} {count:>5}  ({count/len(full_corners):.1%})")

print("\nFree-kick label distribution:")
for label, count in Counter(r["label"] for r in full_freekicks).most_common():
    print(f"  {label:<15} {count:>5}  ({count/len(full_freekicks):.1%})")

print("\nCorners per competition:")
for (comp, season), count in sorted(Counter((r["competition_id"], r["season_id"]) for r in full_corners).items()):
    print(f"  competition={comp:<5} season={season:<5} {count:>5} corners")

Corners: 4467 total, 2468 usable (55.2%)
Free kicks: 1999 total, 1419 usable (71.0%)

Corner label distribution:
  no-shot          2679  (60.0%)
  direct           1124  (25.2%)
  1-pass            227  (5.1%)
  4-plus-pass       190  (4.3%)
  2-pass            151  (3.4%)
  3-pass             96  (2.1%)

Free-kick label distribution:
  no-shot          1103  (55.2%)
  direct            747  (37.4%)
  1-pass             64  (3.2%)
  4-plus-pass        41  (2.1%)
  2-pass             25  (1.3%)
  3-pass             19  (1.0%)

Corners per competition:
  competition=7     season=108     223 corners
  competition=7     season=235     288 corners
  competition=9     season=281     348 corners
  competition=11    season=90      329 corners
  competition=43    season=106     569 corners
  competition=44    season=107      57 corners
  competition=53    season=106     317 corners
  competition=53    season=315     305 corners
  competition=55    season=43      461 corners
  competition=55   

Part B - graph construction

In [39]:
!pip install torch_geometric -q

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv, global_mean_pool
from torch_geometric.loader import DataLoader

import numpy as np
import random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


[notice] A new release of pip is available: 25.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\USER\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch: 2.11.0+cpu
CUDA available: False


define the binary shot label and the graph-construction function

In [40]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

def is_shot_label(label: str) -> int:
    """Collapses our 6-way label into TacticAI-style binary shot/no-shot."""
    return 0 if label == "no-shot" else 1


def build_graph(record: dict):
    """Converts one corner record's freeze_frame into a PyG Data object.
    Returns None if there's no usable freeze frame."""
    freeze_frame = record["freeze_frame"]
    if not freeze_frame:
        return None

    node_features = []
    for player in freeze_frame:
        x, y = player["location"]
        teammate = float(player["teammate"])
        actor = float(player["actor"])
        node_features.append([x / 120.0, y / 80.0, teammate, actor])  # normalize to pitch dims

    n = len(node_features)
    if n < 2:
        return None  # need at least 2 players for a meaningful graph

    x_tensor = torch.tensor(node_features, dtype=torch.float)

    # fully connected graph, like TacticAI (every player can interact with every other)
    edge_index = []
    edge_attr = []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            edge_index.append([i, j])
            same_team = float(freeze_frame[i]["teammate"] == freeze_frame[j]["teammate"])
            edge_attr.append([same_team])

    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    y = torch.tensor([is_shot_label(record["label"])], dtype=torch.float)

    return Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y)

build and inspect one real graph before we build the whole dataset.

In [41]:
full_corners = load_jsonl(FULL_CORNERS_FILE)
usable_corners = [r for r in full_corners if r["has_freeze_frame"]]

print(f"Total corners: {len(full_corners)}, usable: {len(usable_corners)}")

sample_record = usable_corners[0]
sample_graph = build_graph(sample_record)

print(f"\nSample corner: match {sample_record['match_id']}, label={sample_record['label']}")
print(f"Graph: {sample_graph}")
print(f"  Node features (x):\n{sample_graph.x}")
print(f"  Edge index shape: {sample_graph.edge_index.shape}")
print(f"  Edge attr shape: {sample_graph.edge_attr.shape}")
print(f"  Target y: {sample_graph.y}")

Total corners: 4467, usable: 2468

Sample corner: match 3895292, label=direct
Graph: Data(x=[20, 4], edge_index=[2, 380], edge_attr=[380, 1], y=[1])
  Node features (x):
tensor([[0.8074, 0.5394, 1.0000, 0.0000],
        [0.8500, 0.3996, 1.0000, 0.0000],
        [0.8610, 0.4913, 0.0000, 0.0000],
        [0.8745, 0.5354, 1.0000, 0.0000],
        [0.8829, 0.5353, 0.0000, 0.0000],
        [0.8995, 0.4678, 1.0000, 0.0000],
        [0.9044, 0.4863, 0.0000, 0.0000],
        [0.9113, 0.4451, 1.0000, 0.0000],
        [0.9126, 0.4101, 0.0000, 0.0000],
        [0.9131, 0.4636, 0.0000, 0.0000],
        [0.9364, 0.4763, 0.0000, 0.0000],
        [0.9537, 0.4759, 1.0000, 0.0000],
        [0.9622, 0.5110, 1.0000, 0.0000],
        [0.9645, 0.4907, 0.0000, 0.0000],
        [0.9692, 0.5013, 0.0000, 0.0000],
        [0.9721, 0.4426, 0.0000, 0.0000],
        [0.9755, 0.4696, 0.0000, 0.0000],
        [0.9787, 0.4991, 1.0000, 0.0000],
        [0.9891, 0.4909, 0.0000, 0.0000],
        [1.0000, 0.0012, 1.0000,

build the full graph dataset and split train/test

In [42]:
def build_dataset(records):
    graphs = []
    skipped = 0
    for record in records:
        if not record["has_freeze_frame"]:
            continue
        graph = build_graph(record)
        if graph is None:
            skipped += 1
            continue
        graphs.append(graph)
    return graphs, skipped

corner_graphs, n_skipped = build_dataset(full_corners)
print(f"Built {len(corner_graphs)} corner graphs, skipped {n_skipped} (freeze frame present but <2 players)")

label_counts = Counter(int(g.y.item()) for g in corner_graphs)
print(f"Label balance: no-shot={label_counts[0]}, shot={label_counts[1]} "
      f"({label_counts[1]/len(corner_graphs):.1%} positive)")

random.shuffle(corner_graphs)
split_idx = int(0.8 * len(corner_graphs))
train_graphs = corner_graphs[:split_idx]
test_graphs = corner_graphs[split_idx:]

print(f"\nTrain: {len(train_graphs)}, Test: {len(test_graphs)}")

train_loader = DataLoader(train_graphs, batch_size=32, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=32, shuffle=False)

Built 2468 corner graphs, skipped 0 (freeze frame present but <2 players)
Label balance: no-shot=1474, shot=994 (40.3% positive)

Train: 1974, Test: 494


define the GAT model

In [43]:
class SetPieceGAT(nn.Module):
    def __init__(self, node_dim=4, edge_dim=1, hidden_dim=32, heads=4):
        super().__init__()
        self.gat1 = GATv2Conv(node_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.gat1(x, edge_index, edge_attr)
        h = F.elu(h)
        h = self.gat2(h, edge_index, edge_attr)
        h = F.elu(h)

        player_embeddings = h  # per-player (node) embeddings

        graph_embedding = global_mean_pool(h, batch)  # pooled to one vector per corner

        logit = self.classifier(graph_embedding)
        return logit, player_embeddings, graph_embedding


model = SetPieceGAT()
print(model)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")

SetPieceGAT(
  (gat1): GATv2Conv(4, 32, heads=4)
  (gat2): GATv2Conv(128, 32, heads=1)
  (classifier): Linear(in_features=32, out_features=1, bias=True)
)

Total parameters: 10,049


mirror augmentation, applied to the training set only

In [44]:
def mirror_graph(graph):
    """Vertical reflection: flip y-coordinate (left side of goal <-> right side).
    Label, edges, and edge_attr are unchanged -- only node x/y positions flip."""
    x = graph.x.clone()
    x[:, 1] = 1.0 - x[:, 1]  # y was normalized to [0,1]; flip it
    return Data(x=x, edge_index=graph.edge_index.clone(),
                edge_attr=graph.edge_attr.clone(), y=graph.y.clone())


mirrored_train_graphs = [mirror_graph(g) for g in train_graphs]
train_graphs_augmented = train_graphs + mirrored_train_graphs

print(f"Original train set: {len(train_graphs)}")
print(f"After mirror augmentation: {len(train_graphs_augmented)}")
print(f"Test set (unchanged, no augmentation): {len(test_graphs)}")

train_loader = DataLoader(train_graphs_augmented, batch_size=32, shuffle=True)
# test_loader stays as it was -- built from the original, un-augmented test_graphs

Original train set: 1974
After mirror augmentation: 3948
Test set (unchanged, no augmentation): 494


The training loop

In [45]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SetPieceGAT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
criterion = nn.BCEWithLogitsLoss()

N_EPOCHS = 50

def run_epoch(loader, train: bool):
    model.train() if train else model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    for batch in loader:
        batch = batch.to(device)
        if train:
            optimizer.zero_grad()

        logit, _, _ = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        logit = logit.squeeze(-1)
        loss = criterion(logit, batch.y)

        if train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * batch.num_graphs
        preds = (torch.sigmoid(logit) > 0.5).float()
        correct += (preds == batch.y).sum().item()
        total += batch.num_graphs

    return total_loss / total, correct / total


history = []
for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    test_loss, test_acc = run_epoch(test_loader, train=False)
    history.append((epoch, train_loss, train_acc, test_loss, test_acc))

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}  train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
              f"test_loss={test_loss:.4f} test_acc={test_acc:.3f}")

Epoch   1  train_loss=0.6766 train_acc=0.596  test_loss=0.6738 test_acc=0.601
Epoch   5  train_loss=0.6748 train_acc=0.596  test_loss=0.6774 test_acc=0.601
Epoch  10  train_loss=0.6758 train_acc=0.596  test_loss=0.6740 test_acc=0.601
Epoch  15  train_loss=0.6751 train_acc=0.596  test_loss=0.6730 test_acc=0.601
Epoch  20  train_loss=0.6750 train_acc=0.596  test_loss=0.6733 test_acc=0.601
Epoch  25  train_loss=0.6746 train_acc=0.596  test_loss=0.6727 test_acc=0.601
Epoch  30  train_loss=0.6748 train_acc=0.596  test_loss=0.6726 test_acc=0.601
Epoch  35  train_loss=0.6748 train_acc=0.596  test_loss=0.6751 test_acc=0.601
Epoch  40  train_loss=0.6747 train_acc=0.596  test_loss=0.6726 test_acc=0.601
Epoch  45  train_loss=0.6748 train_acc=0.596  test_loss=0.6727 test_acc=0.601
Epoch  50  train_loss=0.6748 train_acc=0.596  test_loss=0.6726 test_acc=0.601


diagnose what the model is actually predicting

In [46]:
model.eval()
all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        logit, _, _ = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        probs = torch.sigmoid(logit.squeeze(-1))
        all_probs.extend(probs.tolist())
        all_preds.extend((probs > 0.5).float().tolist())
        all_labels.extend(batch.y.tolist())

print("Predicted probability stats:")
print(f"  min={min(all_probs):.4f}  max={max(all_probs):.4f}  "
      f"mean={sum(all_probs)/len(all_probs):.4f}  std={np.std(all_probs):.4f}")
print(f"\nPredicted class distribution: {Counter(all_preds)}")
print(f"True label distribution:      {Counter(all_labels)}")

print("\nGradient check on first training batch:")
model.train()
batch = next(iter(train_loader)).to(device)
optimizer.zero_grad()
logit, _, _ = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
loss = criterion(logit.squeeze(-1), batch.y)
loss.backward()
for name, param in model.named_parameters():
    if param.grad is not None:
        print(f"  {name:<30} grad_norm={param.grad.norm().item():.6f}")

Predicted probability stats:
  min=0.4039  max=0.4046  mean=0.4042  std=0.0001

Predicted class distribution: Counter({0.0: 494})
True label distribution:      Counter({0.0: 297, 1.0: 197})

Gradient check on first training batch:
  gat1.att                       grad_norm=0.000022
  gat1.bias                      grad_norm=0.007828
  gat1.lin_l.weight              grad_norm=0.009052
  gat1.lin_l.bias                grad_norm=0.007828
  gat1.lin_r.weight              grad_norm=0.000000
  gat1.lin_r.bias                grad_norm=0.000000
  gat1.lin_edge.weight           grad_norm=0.000000
  gat2.att                       grad_norm=0.000000
  gat2.bias                      grad_norm=0.010510
  gat2.lin_l.weight              grad_norm=0.015564
  gat2.lin_l.bias                grad_norm=0.010510
  gat2.lin_r.weight              grad_norm=0.000000
  gat2.lin_r.bias                grad_norm=0.000000
  gat2.lin_edge.weight           grad_norm=0.000000
  classifier.weight              grad_nor

rebuild the model with LayerNorm and retrain with a higher learning rate

In [47]:
class SetPieceGAT(nn.Module):
    def __init__(self, node_dim=4, edge_dim=1, hidden_dim=32, heads=4):
        super().__init__()
        self.gat1 = GATv2Conv(node_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.gat1(x, edge_index, edge_attr)
        h = self.norm1(h)
        h = F.elu(h)
        h = self.gat2(h, edge_index, edge_attr)
        h = self.norm2(h)
        h = F.elu(h)

        player_embeddings = h
        graph_embedding = global_mean_pool(h, batch)
        logit = self.classifier(graph_embedding)
        return logit, player_embeddings, graph_embedding


model = SetPieceGAT().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)  # raised from 0.001, dropped weight_decay for now
criterion = nn.BCEWithLogitsLoss()

history = []
for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    test_loss, test_acc = run_epoch(test_loader, train=False)
    history.append((epoch, train_loss, train_acc, test_loss, test_acc))
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}  train_loss={train_loss:.4f} train_acc={train_acc:.3f}  "
              f"test_loss={test_loss:.4f} test_acc={test_acc:.3f}")

Epoch   1  train_loss=0.6826 train_acc=0.597  test_loss=0.6725 test_acc=0.601
Epoch   5  train_loss=0.6757 train_acc=0.596  test_loss=0.6735 test_acc=0.601
Epoch  10  train_loss=0.6754 train_acc=0.596  test_loss=0.6725 test_acc=0.601
Epoch  15  train_loss=0.6748 train_acc=0.596  test_loss=0.6725 test_acc=0.601
Epoch  20  train_loss=0.6750 train_acc=0.596  test_loss=0.6732 test_acc=0.601
Epoch  25  train_loss=0.6751 train_acc=0.596  test_loss=0.6725 test_acc=0.601
Epoch  30  train_loss=0.6749 train_acc=0.596  test_loss=0.6726 test_acc=0.601
Epoch  35  train_loss=0.6753 train_acc=0.596  test_loss=0.6730 test_acc=0.601
Epoch  40  train_loss=0.6751 train_acc=0.596  test_loss=0.6725 test_acc=0.601
Epoch  45  train_loss=0.6751 train_acc=0.596  test_loss=0.6726 test_acc=0.601
Epoch  50  train_loss=0.6749 train_acc=0.596  test_loss=0.6725 test_acc=0.601


Receiver grpah builder

In [48]:
MAX_RECEIVER_DIST = 15.0  # StatsBomb units (~meters); beyond this, treat as no reliable match

def build_receiver_graph(corner_event, freeze_frame):
    end_loc = corner_event.get("pass", {}).get("end_location")
    if not end_loc or not freeze_frame:
        return None
    end_loc = np.array(end_loc)

    node_features, teammate_flags, locs = [], [], []
    for p in freeze_frame:
        x, y = p["location"]
        node_features.append([x / 120.0, y / 80.0, float(p["teammate"]), float(p["actor"])])
        teammate_flags.append(p["teammate"])
        locs.append(p["location"])

    n = len(node_features)
    if n < 2:
        return None

    team_idx = [i for i, t in enumerate(teammate_flags) if t]
    if not team_idx:
        return None

    team_locs = np.array([locs[i] for i in team_idx])
    d = np.linalg.norm(team_locs - end_loc, axis=1)
    best = d.argmin()
    if d[best] > MAX_RECEIVER_DIST:
        return None  # no teammate close enough to trust this as the receiver
    receiver_node_idx = team_idx[best]

    x_tensor = torch.tensor(node_features, dtype=torch.float)
    edge_index, edge_attr = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            edge_index.append([i, j])
            edge_attr.append([float(teammate_flags[i] == teammate_flags[j])])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    g = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr)
    g.receiver_idx = torch.tensor([receiver_node_idx], dtype=torch.long)
    g.team_mask = torch.tensor(teammate_flags, dtype=torch.bool)
    return g

build the receiver dataset from the full corners file, and split

In [49]:
receiver_graphs = []
n_skipped_receiver = 0

for record in full_corners:
    if not record["has_freeze_frame"]:
        continue
    # need the raw corner event's pass.end_location -- reconstruct minimal dict from the record
    corner_event_stub = {"pass": {"end_location": record["pass_end_location"]}}
    g = build_receiver_graph(corner_event_stub, record["freeze_frame"])
    if g is None:
        n_skipped_receiver += 1
        continue
    receiver_graphs.append(g)

print(f"Built {len(receiver_graphs)} receiver-labeled graphs")
print(f"Skipped {n_skipped_receiver} (no end_location, too few players, or nearest teammate too far)")

random.shuffle(receiver_graphs)
split_idx = int(0.8 * len(receiver_graphs))
train_receiver_graphs = receiver_graphs[:split_idx]
test_receiver_graphs = receiver_graphs[split_idx:]

print(f"Train: {len(train_receiver_graphs)}, Test: {len(test_receiver_graphs)}")

avg_team_size = np.mean([g.team_mask.sum().item() for g in test_receiver_graphs])
print(f"Avg candidate (teammate) nodes per corner: {avg_team_size:.1f}")
print(f"Random-guess baselines: top1~{1/avg_team_size:.3f}  top3~{3/avg_team_size:.3f}")

Built 2387 receiver-labeled graphs
Skipped 81 (no end_location, too few players, or nearest teammate too far)
Train: 1909, Test: 478
Avg candidate (teammate) nodes per corner: 8.1
Random-guess baselines: top1~0.124  top3~0.372


The receiver-prediction GAT model and training loop

In [50]:
class ReceiverGAT(nn.Module):
    def __init__(self, node_dim=4, edge_dim=1, hidden_dim=32, heads=4):
        super().__init__()
        self.gat1 = GATv2Conv(node_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.node_score = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr):
        h = self.gat1(x, edge_index, edge_attr)
        h = self.norm1(h)
        h = F.elu(h)
        h = self.gat2(h, edge_index, edge_attr)
        h = self.norm2(h)
        h = F.elu(h)
        node_logits = self.node_score(h).squeeze(-1)
        return h, node_logits  # h = per-player embeddings, node_logits = receiver score per node


receiver_model = ReceiverGAT().to(device)
receiver_optimizer = torch.optim.Adam(receiver_model.parameters(), lr=0.005)


def run_receiver_epoch(graphs, train: bool):
    receiver_model.train() if train else receiver_model.eval()
    total_loss, top1, top3, n = 0.0, 0, 0, 0

    for g in graphs:
        g = g.to(device)
        if train:
            receiver_optimizer.zero_grad()

        embeddings, node_logits = receiver_model(g.x, g.edge_index, g.edge_attr)
        team_logits = node_logits[g.team_mask]
        team_indices = torch.nonzero(g.team_mask).squeeze(-1)
        true_pos_in_team = (team_indices == g.receiver_idx.item()).nonzero().item()

        log_probs = F.log_softmax(team_logits, dim=0)
        loss = -log_probs[true_pos_in_team]

        if train:
            loss.backward()
            receiver_optimizer.step()

        total_loss += loss.item()
        ranked = torch.argsort(team_logits, descending=True)
        top1 += int(ranked[0].item() == true_pos_in_team)
        top3 += int(true_pos_in_team in ranked[:3].tolist())
        n += 1

    return total_loss / n, top1 / n, top3 / n

run the training loop and evaluate

In [51]:
N_EPOCHS_RECEIVER = 60

receiver_history = []
for epoch in range(1, N_EPOCHS_RECEIVER + 1):
    train_loss, train_top1, train_top3 = run_receiver_epoch(train_receiver_graphs, train=True)
    test_loss, test_top1, test_top3 = run_receiver_epoch(test_receiver_graphs, train=False)
    receiver_history.append((epoch, train_loss, train_top1, train_top3, test_loss, test_top1, test_top3))

    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}  train_loss={train_loss:.3f} train_top1={train_top1:.3f} train_top3={train_top3:.3f}  "
              f"test_loss={test_loss:.3f} test_top1={test_top1:.3f} test_top3={test_top3:.3f}")

print(f"\nRandom baseline: top1~{1/avg_team_size:.3f}  top3~{3/avg_team_size:.3f}")
print(f"TacticAI's own reported top-3 accuracy (for reference, not a direct comparison): 0.782")

Epoch   1  train_loss=1.853 train_top1=0.277 train_top3=0.664  test_loss=1.658 test_top1=0.303 test_top3=0.749
Epoch   5  train_loss=1.681 train_top1=0.339 train_top3=0.743  test_loss=1.644 test_top1=0.318 test_top3=0.757
Epoch  10  train_loss=1.674 train_top1=0.336 train_top3=0.735  test_loss=1.647 test_top1=0.324 test_top3=0.757
Epoch  15  train_loss=1.670 train_top1=0.343 train_top3=0.742  test_loss=1.636 test_top1=0.331 test_top3=0.764
Epoch  20  train_loss=1.661 train_top1=0.341 train_top3=0.745  test_loss=1.629 test_top1=0.328 test_top3=0.762
Epoch  25  train_loss=1.654 train_top1=0.356 train_top3=0.747  test_loss=1.635 test_top1=0.349 test_top3=0.762
Epoch  30  train_loss=1.651 train_top1=0.359 train_top3=0.744  test_loss=1.621 test_top1=0.331 test_top3=0.766
Epoch  35  train_loss=1.646 train_top1=0.366 train_top3=0.745  test_loss=1.616 test_top1=0.335 test_top3=0.762
Epoch  40  train_loss=1.648 train_top1=0.368 train_top3=0.745  test_loss=1.614 test_top1=0.343 test_top3=0.768
E

get receiver probabilities from the frozen, trained receiver model

In [52]:
receiver_model.eval()  # freeze it -- we're using it for inference only, not retraining it here

def get_receiver_probs(record):
    """Runs the trained receiver_model on one corner's freeze frame, returns a
    per-player probability array (0 for opponents, softmax-over-teammates for teammates)."""
    freeze_frame = record["freeze_frame"]
    node_features, teammate_flags = [], []
    for p in freeze_frame:
        x, y = p["location"]
        node_features.append([x / 120.0, y / 80.0, float(p["teammate"]), float(p["actor"])])
        teammate_flags.append(p["teammate"])

    n = len(node_features)
    x_tensor = torch.tensor(node_features, dtype=torch.float)
    edge_index, edge_attr = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            edge_index.append([i, j])
            edge_attr.append([float(teammate_flags[i] == teammate_flags[j])])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    with torch.no_grad():
        _, node_logits = receiver_model(x_tensor.to(device), edge_index.to(device), edge_attr.to(device))

    probs = torch.zeros(n)
    team_mask = torch.tensor(teammate_flags, dtype=torch.bool)
    team_logits = node_logits[team_mask.to(device)].cpu()
    team_probs = F.softmax(team_logits, dim=0)
    probs[team_mask] = team_probs
    return probs.tolist()

build shot graphs conditioned on receiver probability, and retrain

In [53]:
def build_conditioned_shot_graph(record):
    freeze_frame = record["freeze_frame"]
    receiver_probs = get_receiver_probs(record)

    node_features, teammate_flags = [], []
    for p, r_prob in zip(freeze_frame, receiver_probs):
        x, y = p["location"]
        node_features.append([x / 120.0, y / 80.0, float(p["teammate"]), float(p["actor"]), r_prob])
        teammate_flags.append(p["teammate"])

    n = len(node_features)
    if n < 2:
        return None

    x_tensor = torch.tensor(node_features, dtype=torch.float)
    edge_index, edge_attr = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            edge_index.append([i, j])
            edge_attr.append([float(teammate_flags[i] == teammate_flags[j])])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    y = torch.tensor([is_shot_label(record["label"])], dtype=torch.float)
    return Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y)


conditioned_graphs = []
for record in full_corners:
    if not record["has_freeze_frame"]:
        continue
    g = build_conditioned_shot_graph(record)
    if g is not None:
        conditioned_graphs.append(g)

print(f"Built {len(conditioned_graphs)} receiver-conditioned shot graphs")

random.shuffle(conditioned_graphs)
split_idx = int(0.8 * len(conditioned_graphs))
train_conditioned = conditioned_graphs[:split_idx]
test_conditioned = conditioned_graphs[split_idx:]

mirrored_conditioned = [mirror_graph(g) for g in train_conditioned]
train_conditioned_augmented = train_conditioned + mirrored_conditioned

train_cond_loader = DataLoader(train_conditioned_augmented, batch_size=32, shuffle=True)
test_cond_loader = DataLoader(test_conditioned, batch_size=32, shuffle=False)

print(f"Train (augmented): {len(train_conditioned_augmented)}, Test: {len(test_conditioned)}")

Built 2468 receiver-conditioned shot graphs
Train (augmented): 3948, Test: 494


retrain the shot classifier with the 5th (receiver-conditioned) feature, and evaluate with AUC — not just accuracy, since accuracy alone was misleading before (a model that always predicts "no-shot" gets ~60% "accuracy" for free, which is exactly what fooled us earlier)

In [54]:
from sklearn.metrics import roc_auc_score

class ConditionedShotGAT(nn.Module):
    def __init__(self, node_dim=5, edge_dim=1, hidden_dim=32, heads=4):
        super().__init__()
        self.gat1 = GATv2Conv(node_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, x, edge_index, edge_attr, batch):
        h = self.gat1(x, edge_index, edge_attr)
        h = self.norm1(h); h = F.elu(h)
        h = self.gat2(h, edge_index, edge_attr)
        h = self.norm2(h); h = F.elu(h)
        player_embeddings = h
        graph_embedding = global_mean_pool(h, batch)
        logit = self.classifier(graph_embedding)
        return logit, player_embeddings, graph_embedding


cond_model = ConditionedShotGAT().to(device)
cond_optimizer = torch.optim.Adam(cond_model.parameters(), lr=0.005)
cond_criterion = nn.BCEWithLogitsLoss()

def run_cond_epoch(loader, train: bool):
    cond_model.train() if train else cond_model.eval()
    total_loss, all_probs, all_labels = 0.0, [], []
    for batch in loader:
        batch = batch.to(device)
        if train:
            cond_optimizer.zero_grad()
        logit, _, _ = cond_model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)
        logit = logit.squeeze(-1)
        loss = cond_criterion(logit, batch.y)
        if train:
            loss.backward(); cond_optimizer.step()
        total_loss += loss.item() * batch.num_graphs
        all_probs.extend(torch.sigmoid(logit).detach().cpu().tolist())
        all_labels.extend(batch.y.cpu().tolist())
    auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else float("nan")
    return total_loss / len(all_labels), auc


for epoch in range(1, 51):
    train_loss, train_auc = run_cond_epoch(train_cond_loader, train=True)
    test_loss, test_auc = run_cond_epoch(test_cond_loader, train=False)
    if epoch % 5 == 0 or epoch == 1:
        print(f"Epoch {epoch:>3}  train_loss={train_loss:.4f} train_AUC={train_auc:.3f}  "
              f"test_loss={test_loss:.4f} test_AUC={test_auc:.3f}")

Epoch   1  train_loss=0.6791 train_AUC=0.491  test_loss=0.6780 test_AUC=0.488
Epoch   5  train_loss=0.6748 train_AUC=0.496  test_loss=0.6863 test_AUC=0.468
Epoch  10  train_loss=0.6745 train_AUC=0.495  test_loss=0.6783 test_AUC=0.465
Epoch  15  train_loss=0.6714 train_AUC=0.490  test_loss=0.6800 test_AUC=0.507
Epoch  20  train_loss=0.6720 train_AUC=0.505  test_loss=0.6817 test_AUC=0.476
Epoch  25  train_loss=0.6707 train_AUC=0.507  test_loss=0.6842 test_AUC=0.473
Epoch  30  train_loss=0.6709 train_AUC=0.500  test_loss=0.6803 test_AUC=0.468
Epoch  35  train_loss=0.6707 train_AUC=0.497  test_loss=0.6809 test_AUC=0.459
Epoch  40  train_loss=0.6712 train_AUC=0.507  test_loss=0.6813 test_AUC=0.464
Epoch  45  train_loss=0.6704 train_AUC=0.501  test_loss=0.6812 test_AUC=0.469
Epoch  50  train_loss=0.6717 train_AUC=0.503  test_loss=0.6790 test_AUC=0.468


enriched graph builder (marking distance, distance-to-goal, convex hull compactness, box density)

In [55]:
from scipy.spatial import ConvexHull

def build_enriched_graph(record):
    freeze_frame = record["freeze_frame"]
    locs = np.array([p["location"] for p in freeze_frame])
    teammates = np.array([p["teammate"] for p in freeze_frame])
    actors = np.array([p["actor"] for p in freeze_frame])
    n = len(freeze_frame)

    if teammates.sum() < 3 or (~teammates.astype(bool)).sum() < 1:
        return None  # need enough players on both sides for marking/hull to mean anything

    GOAL = np.array([120.0, 40.0])
    dist_goal = np.linalg.norm(locs - GOAL, axis=1)
    team_locs = locs[teammates.astype(bool)]
    opp_locs = locs[~teammates.astype(bool)]

    mark_dist = np.zeros(n)
    for i in range(n):
        others = opp_locs if teammates[i] else team_locs
        mark_dist[i] = np.linalg.norm(others - locs[i], axis=1).min() if len(others) > 0 else 20.0

    node_feats = np.stack([
        locs[:, 0] / 120.0, locs[:, 1] / 80.0, teammates.astype(float), actors.astype(float),
        dist_goal / 120.0, np.clip(mark_dist, 0, 30) / 30.0
    ], axis=1)

    try:
        hull_area = ConvexHull(team_locs).volume / 8400.0
    except Exception:
        hull_area = 0.0
    n_near_goal = (dist_goal[teammates.astype(bool)] < 12).sum() / max(teammates.sum(), 1)
    graph_feat = torch.tensor([hull_area, n_near_goal], dtype=torch.float).unsqueeze(0)

    x_tensor = torch.tensor(node_feats, dtype=torch.float)
    edge_index, edge_attr = [], []
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            edge_index.append([i, j])
            edge_attr.append([float(teammates[i] == teammates[j])])
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    y = torch.tensor([is_shot_label(record["label"])], dtype=torch.float)
    g = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y)
    g.graph_feat = graph_feat
    return g


enriched_graphs = []
for record in full_corners:
    if not record["has_freeze_frame"]:
        continue
    g = build_enriched_graph(record)
    if g is not None:
        enriched_graphs.append(g)

print(f"Built {len(enriched_graphs)} enriched graphs")
print(f"Positive rate: {np.mean([g.y.item() for g in enriched_graphs]):.3f}")

Built 2460 enriched graphs
Positive rate: 0.402


the enriched GAT model, trained and evaluated with 5-fold cross-validation (more reliable than one train/test split, given how modest and noise-sensitive this signal is)

In [56]:
from sklearn.model_selection import StratifiedKFold

class EnrichedGAT(nn.Module):
    def __init__(self, node_dim=6, edge_dim=1, hidden_dim=32, heads=4, graph_feat_dim=2):
        super().__init__()
        self.gat1 = GATv2Conv(node_dim, hidden_dim, heads=heads, edge_dim=edge_dim, concat=True)
        self.norm1 = nn.LayerNorm(hidden_dim * heads)
        self.gat2 = GATv2Conv(hidden_dim * heads, hidden_dim, heads=1, edge_dim=edge_dim, concat=False)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.classifier = nn.Sequential(nn.Linear(hidden_dim + graph_feat_dim, 16), nn.ReLU(), nn.Linear(16, 1))

    def forward(self, x, edge_index, edge_attr, batch, graph_feat):
        h = self.gat1(x, edge_index, edge_attr)
        h = self.norm1(h); h = F.elu(h)
        h = self.gat2(h, edge_index, edge_attr)
        h = self.norm2(h); h = F.elu(h)
        player_embeddings = h
        graph_embedding = global_mean_pool(h, batch)
        combined = torch.cat([graph_embedding, graph_feat], dim=1)
        logit = self.classifier(combined)
        return logit, player_embeddings, graph_embedding


labels_all = np.array([g.y.item() for g in enriched_graphs])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
idx_arr = np.arange(len(enriched_graphs))

fold_aucs = []
final_fold_models = []  # keep models around so we can pick one for producing final danger scores

for fold, (train_idx, test_idx) in enumerate(skf.split(idx_arr, labels_all)):
    train_g = [enriched_graphs[i] for i in train_idx]
    test_g = [enriched_graphs[i] for i in test_idx]

    mirrored = [mirror_graph(g) for g in train_g]
    for mg, orig_g in zip(mirrored, train_g):
        mg.graph_feat = orig_g.graph_feat.clone()  # graph-level features unaffected by the flip
    train_g_augmented = train_g + mirrored

    train_loader = DataLoader(train_g_augmented, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_g, batch_size=32, shuffle=False)

    fold_model = EnrichedGAT().to(device)
    fold_optimizer = torch.optim.Adam(fold_model.parameters(), lr=0.005, weight_decay=1e-5)
    fold_criterion = nn.BCEWithLogitsLoss()

    for epoch in range(40):
        fold_model.train()
        for batch in train_loader:
            batch = batch.to(device)
            fold_optimizer.zero_grad()
            logit, _, _ = fold_model(batch.x, batch.edge_index, batch.edge_attr, batch.batch, batch.graph_feat)
            loss = fold_criterion(logit.squeeze(-1), batch.y)
            loss.backward()
            fold_optimizer.step()

    fold_model.eval()
    probs, ys = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            logit, _, _ = fold_model(batch.x, batch.edge_index, batch.edge_attr, batch.batch, batch.graph_feat)
            probs.extend(torch.sigmoid(logit.squeeze(-1)).cpu().tolist())
            ys.extend(batch.y.cpu().tolist())

    auc = roc_auc_score(ys, probs)
    fold_aucs.append(auc)
    final_fold_models.append(fold_model)
    print(f"Fold {fold}: AUC={auc:.3f}  (n_train={len(train_g_augmented)}, n_test={len(test_g)})")

print(f"\nMean AUC: {np.mean(fold_aucs):.3f}  (std: {np.std(fold_aucs):.3f})")
print(f"For reference \u2014 TacticAI's own unconditional shot F1 was 0.52 (near chance); "
      f"our position-only AUC of {np.mean(fold_aucs):.3f} reflects a comparable, honestly modest ceiling "
      f"given the absence of velocity/player-identity features.")

Fold 0: AUC=0.500  (n_train=3936, n_test=492)
Fold 1: AUC=0.505  (n_train=3936, n_test=492)
Fold 2: AUC=0.513  (n_train=3936, n_test=492)
Fold 3: AUC=0.562  (n_train=3936, n_test=492)
Fold 4: AUC=0.480  (n_train=3936, n_test=492)

Mean AUC: 0.512  (std: 0.027)
For reference — TacticAI's own unconditional shot F1 was 0.52 (near chance); our position-only AUC of 0.512 reflects a comparable, honestly modest ceiling given the absence of velocity/player-identity features.
